# AutoShotV2 - Heatmap Label Smoothing Training

Datasets:
- `/kaggle/input/datasets/domanh704/heatmap` - source code
- `/kaggle/input/datasets/domanh704/autoshot-datasets3` - Shot videos
- `/kaggle/input/datasets/domanh704/dataset-clipshots` - ClipShots
- *(optional)* output dataset lan truoc de resume

In [ ]:
import os, shutil, subprocess, sys, warnings
from pathlib import Path

warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

# GPU check
r = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                   capture_output=True, text=True, check=False)
GPU_NAME = r.stdout.strip().split('\n')[0] if r.returncode == 0 else ''
print('GPU:', GPU_NAME or 'not detected')

import torch
print(f'torch {torch.__version__}  cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'  {p.name}  VRAM={p.total_memory/1e9:.1f}GB')

# ── Paths co dinh ─────────────────────────────────────────────────────────────
SRC_ROOT       = Path('/kaggle/input/datasets/domanh704/heatmap')
SHOT_ROOT_BASE = Path('/kaggle/input/datasets/domanh704/autoshot-datasets3')
CLIP_ROOT_BASE = Path('/kaggle/input/datasets/domanh704/dataset-clipshots')
RESULT_ROOT    = Path('/kaggle/input/datasets/domanh704/result-heatmap1')  # output lan truoc

# ── Tim SRC_DIR ───────────────────────────────────────────────────────────────
SRC_DIR = None
for init in Path('/kaggle/input').rglob('autoshotv2/__init__.py'):
    root = init.parent.parent.parent
    if (root / 'pyproject.toml').exists():
        SRC_DIR = root; break
    for anc in [root] + list(root.parents)[:3]:
        if (anc / 'pyproject.toml').exists():
            SRC_DIR = anc; break
    if SRC_DIR: break
if SRC_DIR is None:
    for toml in SRC_ROOT.rglob('pyproject.toml'):
        SRC_DIR = toml.parent; break
if SRC_DIR is None:
    raise FileNotFoundError(f'Khong tim thay source autoshotv2 trong {SRC_ROOT}')
print(f'SRC_DIR: {SRC_DIR}')

# ── Tim BASE_CKPT ─────────────────────────────────────────────────────────────
BASE_CKPT = next(Path('/kaggle/input').rglob('ckpt_0_200_0.pth'), None)
if BASE_CKPT is None:
    raise FileNotFoundError('Khong tim thay ckpt_0_200_0.pth')
print(f'BASE_CKPT: {BASE_CKPT}')

# ── Tim Shot root ─────────────────────────────────────────────────────────────
SHOT_ROOT = None
for candidate in SHOT_ROOT_BASE.rglob('ground_truth.txt'):
    if (candidate.parent / 'ads_game_videos').exists():
        SHOT_ROOT = candidate.parent; break
if SHOT_ROOT is None and (SHOT_ROOT_BASE / 'ground_truth.txt').exists():
    SHOT_ROOT = SHOT_ROOT_BASE
if SHOT_ROOT is None:
    raise FileNotFoundError('Khong tim thay Shot dataset')
print(f'SHOT_ROOT: {SHOT_ROOT}')

# ── Tim ClipShots root ────────────────────────────────────────────────────────
CLIPSHOTS_ROOT = None
for candidate in CLIP_ROOT_BASE.rglob('annotations/train.json'):
    if (candidate.parent.parent / 'videos').exists():
        CLIPSHOTS_ROOT = candidate.parent.parent; break
if CLIPSHOTS_ROOT is None and (CLIP_ROOT_BASE / 'annotations' / 'train.json').exists():
    CLIPSHOTS_ROOT = CLIP_ROOT_BASE
if CLIPSHOTS_ROOT is None:
    raise FileNotFoundError('Khong tim thay ClipShots dataset')
print(f'CLIPSHOTS_ROOT: {CLIPSHOTS_ROOT}')

# ── Tim PREV_OUTPUT (result-heatmap1) ─────────────────────────────────────────
PREV_OUTPUT = None
if RESULT_ROOT.exists():
    for marker in ['shot_clipshots_phase2_sample_cache.pkl.parts',
                   'shot_clipshots_phase2_sample_cache.pkl',
                   'shot_clipshots_phase2_sample_cache.pkl.partial.pkl',
                   'checkpoints']:
        found = next(RESULT_ROOT.rglob(marker), None)
        if found:
            PREV_OUTPUT = found.parent if found.is_file() else found.parent
            if found.name == marker and found.is_dir():
                PREV_OUTPUT = found.parent
            break
    if PREV_OUTPUT is None:
        PREV_OUTPUT = RESULT_ROOT
print(f'PREV_OUTPUT: {PREV_OUTPUT or "none (fresh run)"}')

# ── Working dir ───────────────────────────────────────────────────────────────
WORK = Path('/kaggle/working/autoshotv2_run')
WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'checkpoints').mkdir(exist_ok=True)
print(f'WORK: {WORK}')

In [ ]:
# Cai package autoshotv2 vao working dir (input la read-only)
SRC_WORK = Path('/kaggle/working/autoshotv2_pkg')
if not SRC_WORK.exists():
    print(f'Copy source {SRC_DIR} -> {SRC_WORK}')
    shutil.copytree(SRC_DIR, SRC_WORK)
else:
    print(f'Source da co tai {SRC_WORK}')

# 1) Them src/ vao sys.path de kernel hien tai import duoc ngay
SRC_PY = str(SRC_WORK / 'src')
if SRC_PY not in sys.path:
    sys.path.insert(0, SRC_PY)

# 2) Them vao PYTHONPATH de subprocess (smoke/full train) ke thua
existing_pp = os.environ.get('PYTHONPATH', '')
if SRC_PY not in existing_pp:
    os.environ['PYTHONPATH'] = SRC_PY + (':' + existing_pp if existing_pp else '')

# 3) pip install --no-deps de dang ky entry points (autoshotv2-train, v.v.)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(SRC_WORK),
     '--no-deps', '--no-build-isolation', '-q'],
    check=True
)
print('autoshotv2 installed OK')
print(f'sys.path[0]: {sys.path[0]}')
print(f'PYTHONPATH:  {os.environ["PYTHONPATH"]}')

# Kiem tra import ngay tai day
import importlib.util as _ilu
_spec = _ilu.find_spec('autoshotv2')
print(f'autoshotv2 location: {_spec.origin if _spec else "NOT FOUND - loi nghiem trong"}')
if _spec is None:
    raise ImportError(f'Khong tim thay autoshotv2 trong {SRC_PY}. Kiem tra cau truc thu muc.')

# Update SRC_DIR de cac cell sau dung ban writable
SRC_DIR = SRC_WORK

# Restore cache/checkpoint tu lan chay truoc
RESUME_NAMES = [
    'shot_clipshots_phase2_sample_cache.pkl',
    'shot_clipshots_phase2_sample_cache.pkl.partial.pkl',
    'shot_clipshots_phase2_sample_cache.pkl.parts',
    'checkpoints',
]
if PREV_OUTPUT:
    print(f'Restoring from previous output: {PREV_OUTPUT}')
    for name in RESUME_NAMES:
        src_p = PREV_OUTPUT / name
        dst   = WORK / name
        if src_p.exists() and not dst.exists():
            print(f'  restore {name}')
            if src_p.is_dir():
                shutil.copytree(src_p, dst, dirs_exist_ok=True)
            else:
                shutil.copy2(src_p, dst)
    print('Restore done.')

In [ ]:
# Patch ManualAdam trong SRC_WORK de ho tro GradScaler (AMP)
_tp2 = SRC_WORK / 'src' / 'autoshotv2' / 'train_phase2.py'
_code = _tp2.read_text()
if 'param_groups' not in _code:
    _old = ('        self.exp_avg_sq = [torch.zeros_like(p, memory_format=torch.preserve_format)'
            ' for p in self.params]\n')
    _new = _old + '        self.param_groups = [{"params": self.params}]\n'
    if _old in _code:
        _tp2.write_text(_code.replace(_old, _new))
        print('Patched ManualAdam.param_groups OK')
    else:
        print('WARNING: pattern not found, skip patch - kiem tra thu cong')
else:
    print('ManualAdam.param_groups da co, khong can patch')

In [ ]:
# Build metadata pickle tu Shot flat + ClipShots
# Loai 200 video GT goc (gt_scenes_dict_baseline_v2.pickle) ra khoi train
META = WORK / 'shot_clipshots_trainval.pickle'
PREPARE_SCRIPT = SRC_DIR / 'scripts' / 'prepare_shot_clipshots_trainval_flat.py'

# Tim file GT goc (upload vao dataset heatmap cung voi source)
OFFICIAL_GT = next(Path('/kaggle/input').rglob('gt_scenes_dict_baseline_v2.pickle'), None)
if OFFICIAL_GT:
    print(f'Official GT found: {OFFICIAL_GT}')
else:
    print('WARNING: gt_scenes_dict_baseline_v2.pickle khong tim thay - se train co the bi data leak')

if META.exists():
    print('Metadata da co, bo qua.')
else:
    print('Building metadata...')
    cmd = [
        sys.executable, str(PREPARE_SCRIPT),
        '--shot-root',      str(SHOT_ROOT),
        '--clipshots-root', str(CLIPSHOTS_ROOT),
        '--out',            str(META),
        '--val-ratio',      '0.10',
        '--max-val-videos', '200',
        '--seed',           '42',
    ]
    if OFFICIAL_GT:
        cmd += ['--official-gt', str(OFFICIAL_GT)]
    subprocess.run(cmd, check=True)

import pickle
with META.open('rb') as f:
    meta = pickle.load(f)
print(f"Train          : {len(meta['train_keys'])}")
print(f"Val            : {len(meta['val_keys'])}")
print(f"Shot test (orig): {len(meta['shot_test_entries'])}")
print(f"Official test  : {len(meta.get('official_test_entries', {}))}")

In [ ]:
# ══════════════════════════════════════════════════════════════
# PREFLIGHT CHECK — chay nhanh, khong can GPU, phat hien loi som
# ══════════════════════════════════════════════════════════════
import importlib, traceback

errors = []

# 1. Import autoshotv2
try:
    import autoshotv2
    print(f'[OK] import autoshotv2  ({autoshotv2.__file__})')
except Exception as e:
    errors.append(f'[FAIL] import autoshotv2: {e}')

# 2. Kiem tra cac module chinh
for mod in ['autoshotv2.train_phase2', 'autoshotv2.phase2_data',
            'autoshotv2.eval', 'autoshotv2.common']:
    try:
        importlib.import_module(mod)
        print(f'[OK] {mod}')
    except Exception as e:
        errors.append(f'[FAIL] {mod}: {e}')
        traceback.print_exc()

# 3. Kiem tra paths
checks = {
    'BASE_CKPT': BASE_CKPT,
    'SHOT_ROOT/ground_truth.txt': SHOT_ROOT / 'ground_truth.txt',
    'CLIPSHOTS/annotations/train.json': CLIPSHOTS_ROOT / 'annotations' / 'train.json',
    'CLIPSHOTS/videos/train': CLIPSHOTS_ROOT / 'videos' / 'train',
    'META pickle': META,
    'prepare_script': SRC_DIR / 'scripts' / 'prepare_shot_clipshots_trainval_flat.py',
}
for name, path in checks.items():
    if Path(path).exists():
        print(f'[OK] {name}: {path}')
    else:
        errors.append(f'[FAIL] {name} NOT FOUND: {path}')

# 4. Kiem tra make_heatmap_labels
try:
    import numpy as np
    from autoshotv2.phase2_data import make_heatmap_labels
    one_hot = np.zeros(100, dtype=np.float32)
    one_hot[50] = 1.0
    lab = make_heatmap_labels(one_hot, sigma=3.0, mode='gaussian')
    assert abs(lab[50] - 1.0) < 1e-5, 'center should be 1.0'
    assert lab[53] < 1.0, 'should decay'
    assert lab[60] < 0.01, 'should be near 0 far away'
    print(f'[OK] make_heatmap_labels  center={lab[50]:.3f} +3={lab[53]:.3f} +9={lab[59]:.3f}')
except Exception as e:
    errors.append(f'[FAIL] make_heatmap_labels: {e}')
    traceback.print_exc()

# 5. Kiem tra FocalLoss voi smoothing
try:
    import torch
    from autoshotv2.train_phase2 import FocalLoss
    loss_fn = FocalLoss(gamma=2.0, alpha=0.6, smoothing=0.1, asymmetric=True)
    pred = torch.tensor([0.7, 0.3, 0.9])
    tgt  = torch.tensor([1.0, 0.0, 1.0])
    loss = loss_fn(pred, tgt)
    print(f'[OK] FocalLoss(smoothing=0.1)  loss={loss.item():.4f}')
except Exception as e:
    errors.append(f'[FAIL] FocalLoss: {e}')
    traceback.print_exc()

# ── Ket qua preflight ──────────────────────────────────────────
print()
if errors:
    print('=== PREFLIGHT FAILED ===')
    for e in errors:
        print(' ', e)
    raise RuntimeError(f'{len(errors)} loi preflight - sua truoc khi chay smoke/full train')
else:
    print('=== PREFLIGHT PASSED — san sang smoke train ===')

In [ ]:
# ══════════════════════════════════════════════════════════════
# SMOKE TRAIN — 3 video, 2 epoch, kiem tra pipeline GPU end-to-end
# ══════════════════════════════════════════════════════════════
r = subprocess.run([
    sys.executable, '-m', 'autoshotv2.train_phase2',
    '--meta',                  str(META),
    '--base-ckpt',             str(BASE_CKPT),
    '--epochs',                '2',
    '--max-train-videos',      '3',
    '--max-total-samples',     '300',
    '--max-samples-per-video', '100',
    '--max-val-videos',        '2',
    '--max-test-videos',       '2',
    '--batch-size',            '64',
    '--loss',                  'focal',
    '--gamma',                 '2.0',
    '--alpha',                 '0.6',
    '--lr',                    '7e-6',
    '--manyhot-weight',        '0.3',
    '--heatmap-sigma',         '3.0',
    '--heatmap-mode',          'gaussian',
    '--label-smoothing',       '0.1',
    '--ls-asymmetric',
    '--amp',
    '--sample-cache',          str(WORK / 'smoke_cache.pkl'),
    '--out-ckpt',              str(WORK / 'smoke_ckpt.pth'),
    '--results',               str(WORK / 'smoke_results.pkl'),
    '--eval-cache-dir',        str(WORK / 'smoke_eval'),
    '--resume-state',          str(WORK / 'smoke_resume.pt'),
    '--checkpoint-dir',        str(WORK / 'smoke_checkpoints'),
    '--rebuild-sample-cache',
    '--no-eval-cache',
    '--no-resume',
    '--seed', '42',
], check=False)

if r.returncode == 0:
    print('\n=== SMOKE TRAIN PASSED — san sang full train ===')
else:
    raise RuntimeError(f'Smoke train FAILED (returncode={r.returncode}) — xem log o tren')

In [ ]:
# Full training — 30 epoch, resume tu checkpoint neu co
#
# Hyperparams tot nhat tu focal sweep (autoshotv2v8):
#   gamma=2.0, alpha=0.6, manyhot_weight=0.3, lr=7e-6
# Them moi: heatmap-sigma=3.0, label-smoothing=0.1, AMP fp16
# Data: loai 200 video GT goc khoi train (fix data leakage)

RESUME_STATE = WORK / 'checkpoints' / 'phase2_resume.pt'

r = subprocess.run([
    sys.executable, '-m', 'autoshotv2.train_phase2',
    '--meta',                  str(META),
    '--base-ckpt',             str(BASE_CKPT),
    '--out-ckpt',              str(WORK / 'ckpt_phase2_best.pth'),
    '--sample-cache',          str(WORK / 'shot_clipshots_phase2_sample_cache.pkl'),
    '--results',               str(WORK / 'results.pkl'),
    '--resume-state',          str(RESUME_STATE),
    '--checkpoint-dir',        str(WORK / 'checkpoints'),
    '--eval-cache-dir',        str(WORK / 'eval_cache'),
    # Hyperparams
    '--epochs',                '30',
    '--batch-size',            '1024',
    '--loss',                  'focal',
    '--gamma',                 '2.0',
    '--alpha',                 '0.6',
    '--lr',                    '7e-6',
    '--weight-decay',          '1e-4',
    '--manyhot-weight',        '0.3',
    # Heatmap + label smoothing
    '--heatmap-sigma',         '3.0',
    '--heatmap-mode',          'gaussian',
    '--label-smoothing',       '0.1',
    '--ls-asymmetric',
    # Sampling
    '--max-samples-per-video', '160',
    '--neg-per-pos',           '3',
    '--boundary-window',       '1',
    '--seed',                  '42',
    '--data-seed',             '42',
    # T4 fp16
    '--amp',
    # Checkpoint moi 5 epoch de resume neu het gio
    '--save-every-epochs',     '5',
    '--log-every-batches',     '50',
    '--max-val-videos',        '200',
    # Dung an toan truoc 11h Kaggle (620 phut = 10h20)
    '--stop-after-minutes',    '620',
], check=False)

if r.returncode == 0:
    print('Training hoan thanh day du 30 epoch.')
else:
    print(f'returncode={r.returncode} — het gio hoac loi.')
    print('Luu output thanh dataset, attach vao lan chay tiep de resume tu checkpoint.')

In [ ]:
import pickle, sys, os
from pathlib import Path
import numpy as np
from scipy.special import expit
from scipy.ndimage import gaussian_filter1d

WORK = Path('/kaggle/working/autoshotv2_run')

# ── Ham eval chuan goc AutoShot (copy tu utils.py, khong can ffmpeg) ──────────
def predictions_to_scenes(predictions):
    scenes = []
    t, t_prev, start = -1, 0, 0
    for i, t in enumerate(predictions):
        if t_prev == 1 and t == 0: start = i
        if t_prev == 0 and t == 1 and i != 0: scenes.append([start, i])
        t_prev = t
    if t == 0: scenes.append([start, i])
    if len(scenes) == 0: return np.array([[0, len(predictions)-1]], dtype=np.int32)
    return np.array(scenes, dtype=np.int32)

def evaluate_scenes(gt_scenes, pred_scenes, n_frames_miss_tolerance=2):
    shift = n_frames_miss_tolerance / 2
    gt_s   = gt_scenes.astype(np.float32)   + np.array([[-0.5+shift, 0.5-shift]])
    pred_s = pred_scenes.astype(np.float32) + np.array([[-0.5+shift, 0.5-shift]])
    gt_t   = np.stack([gt_s[:-1,1],   gt_s[1:,0]],   1)
    pred_t = np.stack([pred_s[:-1,1], pred_s[1:,0]], 1)
    i = j = tp = fp = fn = 0
    while i < len(gt_t) or j < len(pred_t):
        if   j == len(pred_t):                          fn+=1; i+=1
        elif i == len(gt_t):                            fp+=1; j+=1
        elif pred_t[j,1] < gt_t[i,0]:                  fp+=1; j+=1
        elif pred_t[j,0] > gt_t[i,1]:                  fn+=1; i+=1
        else:                                           tp+=1; i+=1; j+=1
    p = tp/(tp+fp) if tp+fp else 0
    r = tp/(tp+fn) if tp+fn else 0
    f1 = 2*p*r/(p+r) if p+r else 0
    return p, r, f1, (tp, fp, fn)

def mAP_f1_max(one_hot_pred, gt_scenes):
    """Tim threshold toi uu de max F1."""
    best = (0, 0, 0, 0)
    for thr in np.arange(0.05, 0.96, 0.05):
        tp = fp = fn = 0
        for name, pred in one_hot_pred.items():
            pred_sc = predictions_to_scenes((pred > thr).astype(np.uint8))
            _, _, _, (tp_, fp_, fn_) = evaluate_scenes(gt_scenes[name], pred_sc)
            tp+=tp_; fp+=fp_; fn+=fn_
        p = tp/(tp+fp) if tp+fp else 0
        r = tp/(tp+fn) if tp+fn else 0
        f1 = 2*p*r/(p+r) if p+r else 0
        if f1 > best[0]: best = (f1, p, r, thr)
    return best  # (f1, p, r, thr)

# ── 1. In ket qua training ────────────────────────────────────────────────────
results_pkl = WORK / 'results.pkl'
if not results_pkl.exists():
    results_pkl = next(Path('/kaggle/input').rglob('results.pkl'), None)

if results_pkl:
    res = pickle.loads(Path(results_pkl).read_bytes())
    print('=== KET QUA TRAINING ===')
    for k, v in res.items():
        if isinstance(v, dict) and 'f1' in v:
            print(f'  [{k}] F1={v["f1"]:.4f}  P={v.get("precision",0):.4f}'
                  f'  R={v.get("recall",0):.4f}'
                  + (f'  thr={v["threshold"]:.3f}' if 'threshold' in v else ''))

# ── 2. Eval tren 200 video GT goc (chuan tac gia) ────────────────────────────
print('\n=== EVAL CHINH THUC THEO CHUAN TAC GIA GOC ===')

# Tim GT
OFFICIAL_GT = next(Path('/kaggle/input').rglob('gt_scenes_dict_baseline_v2.pickle'), None)
if not OFFICIAL_GT:
    print('Khong tim thay gt_scenes_dict_baseline_v2.pickle'); raise SystemExit

with open(OFFICIAL_GT, 'rb') as f:
    gt_scenes_dict = pickle.load(f)
print(f'GT videos: {len(gt_scenes_dict)}')

# Tim checkpoint de lay postprocess config
import torch
CKPT = WORK / 'ckpt_phase2_best.pth'
if not CKPT.exists():
    CKPT = next(Path('/kaggle/input').rglob('ckpt_phase2_best.pth'), None)
ckpt_data = torch.load(CKPT, map_location='cpu')
temperature = float(ckpt_data.get('temperature', 1.0))
sigma       = float(ckpt_data.get('sigma', 0.0))
print(f'Postprocess: temperature={temperature:.3f}  sigma={sigma:.2f}')

# Tim logits
EVAL_CACHE  = WORK / 'eval_cache'
logits_file = next(
    (p for p in [EVAL_CACHE/'shot_test_logits.pkl',
                 EVAL_CACHE/'official_test_logits.pkl']
     if p.exists()),
    next(Path('/kaggle/input').rglob('shot_test_logits.pkl'), None)
)
if not logits_file:
    print('Khong tim thay logits file'); raise SystemExit

with open(logits_file, 'rb') as f:
    logits_data = pickle.load(f)
logits_dict = logits_data['logits']

# Chuyen logits -> probs va match voi GT
pred_dict = {}
for key, logits in logits_dict.items():
    name = key.split(':', 1)[-1]
    if name not in gt_scenes_dict: continue
    l = logits.squeeze(-1).astype(np.float32) / temperature
    probs = expit(l)
    if sigma > 0: probs = gaussian_filter1d(probs, sigma=sigma)
    pred_dict[name] = probs

gt_filtered = {k: v for k, v in gt_scenes_dict.items() if k in pred_dict}
print(f'Video matched: {len(pred_dict)}/200')

if len(pred_dict) == 0:
    print('Khong co video nao match — logits dung cho original_videos, GT dung cho video_download.')
    print('Can chay inference rieng tren 200 video GT de co ket qua chinh thuc.')
else:
    f1, p, r, thr = mAP_f1_max(pred_dict, gt_filtered)
    print(f'\n=== KET QUA TREN {len(pred_dict)} VIDEO TEST (CHUAN TAC GIA) ===')
    print(f'AutoShotV2 heatmap+smoothing: F1={f1:.4f}  P={p:.4f}  R={r:.4f}  thr={thr:.2f}')
    print(f'\n=== SO SANH VOI PAPER ===')
    print(f'TransNetV2 baseline       : F1=0.7993  P=0.9042  R=0.7162')
    print(f'AutoShot supernet (paper) : F1=0.8405  P=0.8473  R=0.8339')
    print(f'AutoShotV2 (cua chung ta) : F1={f1:.4f}  P={p:.4f}  R={r:.4f}')

# ── 3. Files output ───────────────────────────────────────────────────────────
print('\n=== FILES OUTPUT ===')
for p in sorted(WORK.rglob('*')):
    if p.is_file() and '.parts/' not in str(p):
        print(f'  {p.relative_to(WORK)}  ({p.stat().st_size/1e6:.1f} MB)')